In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
#from matplotlib.ticker import MaxNLocator
from scipy.stats import mannwhitneyu

# Centrosome analysis

Channel 1 (RED) - pericentrin
Channel 2 (GREEN) - gamma-tubulin

## Parameters

In [ ]:
DIR = "/mnt/c/users/helen/Desktop/test"
THRESHOLD = 3 # % cells with >=3 objects
PARAMETER = "centrosomes"

## Data loading

In [ ]:
from utils import load_data, data_subset

In [ ]:
df = load_data(DIR)

In [ ]:
df.tail()

In [ ]:
ch1 = data_subset(df, channel = 1)
ch2 = data_subset(df, channel = 2)

## Data processing

In [ ]:
roi_counts = (
    ch2.groupby(["Image", "Cell"])["ROI"]
       .count()
       .reset_index(name="Objects_number")
)

In [ ]:
roi_counts

# TEST

In [ ]:
data = {
    "Image": [
        "2_hct116_wt_pcdna_vacio_+_aph1a_g4__pcnt-gamma_tub-07",
        "2_hct116_wt_pcdna_vacio_+_aph1a_g4__pcnt-gamma_tub-07",
        "2_hct116_wt_pcdna_vacio_+_aph1a_g4__pcnt-gamma_tub-07",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-04",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-04",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-04",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-05",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-05",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-05",
        "2_hct116_wt_pcdna_vacio__pcnt-gamma_tub-05",
        "2_hct116_wt_aph1a_g4__pcnt-gamma_tub-01",
        "2_hct116_wt_aph1a_g4__pcnt-gamma_tub-01",
        "2_hct116_wt_aph1a_g4__pcnt-gamma_tub-01",
        "2_hct116_wt_aph1a_g4__pcnt-gamma_tub-02",
        "2_hct116_wt_aph1a_g4__pcnt-gamma_tub-02",
    ],

    "Cell": [
        1, 2, 3,
        1, 2, 3,
        1, 2, 3, 4,
        1, 2, 3,
        1, 2
    ],

    "Objects_number": [
        3, 2, 4,
        1, 2, 2,
        2, 3, 1, 2,
        4, 3, 5,
        2, 4
    ]
}

df = pd.DataFrame(data)

In [ ]:
# Add sample column manually
df['Sample'] = df['Image'].apply(lambda x: 'sample1' if 'pcdna_vacio' in x else 'sample2')

In [ ]:
df.head()

In [ ]:
# Empty list
results_dicts = []

# Iteration through different samples
for sample in df['Sample'].unique():
    sb = df[df['Sample'] == sample]
    
    # Calculate the persentage of cells with # objects >= threshold
    n_total = len(sb)
    above_thresh = sum(sb['Objects_number'] >= THRESHOLD)
    percentage = 100 * above_thresh/n_total
    sd = np.sqrt(percentage * (100  - percentage)) # SD Bernoulli? 
    
    temp = {
            'Sample_name': sample,
            'Above_threshold': above_thresh,
            'N_total_cells': n_total,
            'Percentage': percentage,
            'SD': sd,
        }
    
    results_dicts.append(temp)


results = pd.DataFrame(results_dicts)

In [ ]:
results

## Graphs

In [ ]:
from utils import barplot_normal

In [ ]:
barplot_normal(df = results)